In [14]:
from scrape import scrape_pdfs
from extract import process_pdfs

IMMIGRANT_URL = "https://travel.state.gov/content/travel/en/legal/visa-law0/visa-statistics/immigrant-visa-statistics/monthly-immigrant-visa-issuances.html"

links = scrape_pdfs(IMMIGRANT_URL, ["FSC"])

test_df = process_pdfs([links[0]], "immigrant")

print(links[0])
print(test_df.head())
print(test_df.columns)
print(len(test_df))


https://travel.state.gov/content/dam/visas/Statistics/Immigrant-Statistics/MonthlyIVIssuances/March%202017%20-%20IV%20Issuances%20by%20FSC%20and%20Visa%20Class%20-%20Worldwide.pdf
       country visa_type count        date visa_program
0  Afghanistan       CR1    11  March 2017    immigrant
1  Afghanistan       DV1     2  March 2017    immigrant
2  Afghanistan       DV2     1  March 2017    immigrant
3  Afghanistan       F31     1  March 2017    immigrant
4  Afghanistan       F41     1  March 2017    immigrant
Index(['country', 'visa_type', 'count', 'date', 'visa_program'], dtype='object')
2438


In [18]:
from transform import standardize_countries, split_date
from scrape import scrape_pdfs
from extract import process_pdfs


# Test the date transformation fix
original = process_pdfs([links[0]], "immigrant")
transformed = standardize_countries(original.copy())
split_dates = split_date(transformed.copy())

# Check for NaT values (dates that couldn't be parsed)
print("Date columns sample:")
print(split_dates[["date", "year", "month"]].head(10))

print(f"\nTotal rows: {len(split_dates)}")
print(f"NaT values in date column: {split_dates['date'].isna().sum()}")
print(f"Successfully parsed dates: {split_dates['date'].notna().sum()}")


Date columns sample:
        date  year  month
0 2017-03-01  2017      3
1 2017-03-01  2017      3
2 2017-03-01  2017      3
3 2017-03-01  2017      3
4 2017-03-01  2017      3
5 2017-03-01  2017      3
6 2017-03-01  2017      3
7 2017-03-01  2017      3
8 2017-03-01  2017      3
9 2017-03-01  2017      3

Total rows: 2438
NaT values in date column: 0
Successfully parsed dates: 2438


In [17]:

# Test non-immigrant file
from transform import standardize_countries, split_date

NONIMMIGRANT_URL = "https://travel.state.gov/content/travel/en/legal/visa-law0/visa-statistics/nonimmigrant-visa-statistics/monthly-nonimmigrant-visa-issuances.html"

non_links = scrape_pdfs(NONIMMIGRANT_URL, ["nationality"])

non_original = process_pdfs([non_links[0]], "nonimmigrant")
non_transformed = standardize_countries(non_original.copy())
non_split = split_date(non_transformed.copy())

print("Non-immigrant date columns sample:")
print(non_split[["date", "year", "month"]].head(10))

print(f"\nTotal rows: {len(non_split)}")
print(f"NaT values in date column: {non_split['date'].isna().sum()}")
print(f"Successfully parsed dates: {non_split['date'].notna().sum()}")


KeyError: 'country'

In [23]:

# Check the raw date values before and after transformation
print("Original date column (raw strings):")
print(original["date"].unique()[:10])
print(f"Data type: {original['date'].dtype}")

print("\n\nAfter split_date transformation:")
print(split_dates["date"].unique()[:10])
print(f"Data type: {split_dates['date'].dtype}")

print("\n\nChecking for mixed formats in date representations:")
print(split_dates["date"].astype(str).unique()[:10])


Original date column (raw strings):
['March 2017']
Data type: object


After split_date transformation:
<DatetimeArray>
['2017-03-01 00:00:00']
Length: 1, dtype: datetime64[ns]
Data type: datetime64[ns]


Checking for mixed formats in date representations:
['2017-03-01']


In [22]:

# Test CSV write/read behavior
import pandas as pd
import tempfile
import os

# Write to CSV
with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False) as f:
    temp_csv = f.name
    split_dates.to_csv(temp_csv, index=False)

# Read back
read_back = pd.read_csv(temp_csv)
print("Date column after CSV write/read:")
print(read_back["date"].unique()[:10])
print(f"Data type: {read_back['date'].dtype}")

print("\n\nString representation:")
print(read_back["date"].astype(str).unique()[:10])

os.remove(temp_csv)


Date column after CSV write/read:
['2017-03-01']
Data type: object


String representation:
['2017-03-01']


In [24]:

# Check existing CSV for mixed date formats
existing_csv = pd.read_csv("../data/processed/visa_data.csv")
print("Existing visa_data.csv date values:")
print(existing_csv["date"].unique()[:20])
print(f"\nTotal unique dates: {existing_csv['date'].nunique()}")
print(f"Data type: {existing_csv['date'].dtype}")

# Check if there are mixed formats
date_formats = existing_csv["date"].astype(str).apply(lambda x: x.split(' ')[1] if len(x.split(' ')) > 1 else "date_only")
print(f"\nDate format distribution:")
print(date_formats.value_counts())


Existing visa_data.csv date values:
['2017-03-01' '2017-04-01' '2017-05-01' '2017-06-01' '2017-07-01'
 '2017-08-01' '2017-09-01' '2017-10-01' '2017-11-01' '2017-12-01'
 '2018-01-01' '2018-02-01' '2018-03-01' '2018-04-01' '2018-05-01'
 '2018-06-01' '2018-07-01' '2018-08-01' '2018-09-01' '2018-10-01']

Total unique dates: 155
Data type: object

Date format distribution:
date
00:00:00     482358
date_only    109445
Name: count, dtype: int64


/tmp/ipykernel_10834/2935996641.py:2: DtypeWarning: Columns (7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  existing_csv = pd.read_csv("../data/processed/visa_data.csv")


In [ ]:
%pip install geopandas
import geopandas as gpd
import geodatasets



Note: you may need to restart the kernel to use updated packages.


AttributeError: The geopandas.dataset has been deprecated and was removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.

In [5]:
world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
world["name"]


AttributeError: The geopandas.dataset has been deprecated and was removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.